In [ ]:
# Dynamic Guardian — Stepwise Colab Pipeline
# STEP 1 — Setup & Imports

# (Optional) Pin TensorFlow version for compatibility
# Uncomment this line if you want to ensure TF 2.11.0
# !pip install -q tensorflow==2.11.0

import os, json, zipfile
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Configurations
CSV_PATH = '/content/master_dataset_3000.csv'  # Upload this file to Colab
OUTPUT_DIR = '/content/dynamic_guardian_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


In [ ]:
# STEP 2 — Load dataset & inspect
print('Loading dataset...')
df = pd.read_csv(CSV_PATH)
print('Rows, Cols:', df.shape)
print('Columns:', df.columns.tolist())


Loading dataset...
Rows, Cols: (3000, 24)
Columns: ['TIMESTAMP', 'DEVICEID', 'LAT', 'LON', 'CO2_PPM', 'CABIN_TEMPERATURE_C', 'AMBIENT_NOISE_DB_AVG', 'VEHICLE_SPEED', 'ACCELEROMETER_Y', 'ACCELEROMETER_Z', 'STEERING_WHEEL_ANGLE', 'ACCELERATOR_PEDAL_POSITION', 'BRAKE_PEDAL_PRESSURE', 'cell_voltage_v', 'cell_temperature_c', 'module_current_a', 'soc_pct', 'soh_pct', 'HEART_RATE_BPM', 'HEART_RATE_VARIABILITY_HRV', 'FATIGUE_ALERT', 'STRESS_ALERT', 'IS_HARSH_BRAKING', 'IS_POTHOLE']


In [ ]:
# STEP 3 — Define feature order & map labels

FEATURE_ORDER = [
    'HEART_RATE_BPM','HEART_RATE_VARIABILITY_HRV','HEART_RATE_BPM_STD',
    'STEERING_WHEEL_ANGLE','ACCELERATOR_PEDAL_POSITION','BRAKE_PEDAL_PRESSURE',
    'STEERING_WHEEL_ANGLE_STD','BRAKE_PEDAL_PRESSURE_DELTA','CO2_PPM',
    'CABIN_TEMPERATURE_C','AMBIENT_NOISE_DB_AVG','soc_pct','cell_voltage_v',
    'ACCELEROMETER_Z','VEHICLE_SPEED','ACCELEROMETER_Y',
    'cell_temperature_c','module_current_a','soh_pct'
]

LABEL_COLUMNS = ['FATIGUE_ALERT','STRESS_ALERT','IS_HARSH_BRAKING','IS_POTHOLE']

def resolve_label(row):
    if row.get('STRESS_ALERT',0)==1: return 'stress'
    if row.get('FATIGUE_ALERT',0)==1: return 'fatigue'
    if row.get('IS_HARSH_BRAKING',0)==1: return 'harsh_brake'
    if row.get('IS_POTHOLE',0)==1: return 'pothole'
    return 'normal'

print('\nGenerating single-label column...')
df['label'] = df.apply(resolve_label, axis=1)
print(df['label'].value_counts())



Generating single-label column...
label
normal     1950
stress      450
fatigue     300
pothole     300
Name: count, dtype: int64


In [ ]:
# STEP 4 — Feature engineering & safety checks

if 'HEART_RATE_BPM_STD' not in df.columns:
    df['HEART_RATE_BPM_STD'] = df['HEART_RATE_BPM'].rolling(5,min_periods=1).std().fillna(0)

if 'STEERING_WHEEL_ANGLE_STD' not in df.columns:
    df['STEERING_WHEEL_ANGLE_STD'] = df['STEERING_WHEEL_ANGLE'].rolling(5,min_periods=1).std().fillna(0)

if 'BRAKE_PEDAL_PRESSURE_DELTA' not in df.columns:
    df['BRAKE_PEDAL_PRESSURE_DELTA'] = df['BRAKE_PEDAL_PRESSURE'].diff().fillna(0)

missing = [c for c in FEATURE_ORDER if c not in df.columns]
if missing:
    raise ValueError('Missing required features: ' + ', '.join(missing))

X = df[FEATURE_ORDER].astype(float).to_numpy()
labels = sorted(df['label'].unique())
label_to_idx = {lab:i for i,lab in enumerate(labels)}
print('Labels mapped:', label_to_idx)
y = df['label'].map(label_to_idx).to_numpy()


Labels mapped: {'fatigue': 0, 'normal': 1, 'pothole': 2, 'stress': 3}


In [ ]:
# STEP 5 — Split and scale data

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler parameters
scaler_json = {
    'feature_order': FEATURE_ORDER,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist()
}
with open(os.path.join(OUTPUT_DIR, 'scaling_parameters.json'), 'w') as f:
    json.dump(scaler_json, f, indent=2)

print('Saved scaling_parameters.json to', OUTPUT_DIR)


Saved scaling_parameters.json to /content/dynamic_guardian_output


In [ ]:
# STEP 6 — Build and train compact neural network

num_features = X_train_scaled.shape[1]
num_classes = len(labels)

model = keras.Sequential([
    layers.Input(shape=(num_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(32, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(X_train_scaled, y_train,
                    validation_data=(X_test_scaled, y_test),
                    epochs=100, batch_size=64, callbacks=[es], verbose=2)

saved_model_dir = os.path.join(OUTPUT_DIR, 'dynamic_guardian_saved_model')
model.export(saved_model_dir)
print('Saved TF SavedModel to', saved_model_dir)


Epoch 1/100
38/38 - 3s - 81ms/step - accuracy: 0.7667 - loss: 0.8916 - val_accuracy: 0.9700 - val_loss: 0.4140
Epoch 2/100
38/38 - 0s - 8ms/step - accuracy: 0.9821 - loss: 0.2204 - val_accuracy: 1.0000 - val_loss: 0.0782
Epoch 3/100
38/38 - 0s - 8ms/step - accuracy: 0.9975 - loss: 0.0545 - val_accuracy: 1.0000 - val_loss: 0.0258
Epoch 4/100
38/38 - 1s - 19ms/step - accuracy: 0.9996 - loss: 0.0230 - val_accuracy: 1.0000 - val_loss: 0.0129
Epoch 5/100
38/38 - 0s - 12ms/step - accuracy: 0.9996 - loss: 0.0129 - val_accuracy: 1.0000 - val_loss: 0.0078
Epoch 6/100
38/38 - 1s - 15ms/step - accuracy: 1.0000 - loss: 0.0086 - val_accuracy: 1.0000 - val_loss: 0.0052
Epoch 7/100
38/38 - 0s - 9ms/step - accuracy: 1.0000 - loss: 0.0062 - val_accuracy: 1.0000 - val_loss: 0.0037
Epoch 8/100
38/38 - 0s - 7ms/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 9/100
38/38 - 0s - 8ms/step - accuracy: 1.0000 - loss: 0.0036 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 

In [ ]:
# STEP 7 — Zip SavedModel & Quantize to TFLite

zip_path = os.path.join(OUTPUT_DIR, 'saved_model_v10.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(saved_model_dir):
        for fname in files:
            zf.write(os.path.join(root, fname),
                     os.path.relpath(os.path.join(root, fname), saved_model_dir))

print('Zipped model:', zip_path)

# Representative dataset
rep_samples = X_train_scaled[:100]
def representative_data_gen():
    for i in range(rep_samples.shape[0]):
        yield [rep_samples[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

try:
    tflite_model = converter.convert()
    tflite_path = os.path.join(OUTPUT_DIR, 'dynamic_guardian_int8.tflite')
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    print('Saved int8 TFLite to', tflite_path)
except Exception as e:
    print('TFLite conversion failed:', e)


Zipped model: /content/dynamic_guardian_output/saved_model_v10.zip
Saved int8 TFLite to /content/dynamic_guardian_output/dynamic_guardian_int8.tflite
